# Synthetic Data Generator
### Step 9 : Row Rebuilder

In [1]:
from utils.postgres_util import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.pipeline.row_rebuilder import (
    rebuild_consumed_messages_to_observations,
)

In [2]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME = str("synthetic_sensor_messages_consumed_stage")
REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME = str("synthetic_sensor_observations_rebuilt_stage")


REBUILD_STATUS = str("pending")
NUMBER_OF_SENSORS = int(52)

COMPLETE_ONLY_FLAG = True
MARK_SOURCE_REBUILT_FLAG = True

OBSERVATION_WINDOW_SIZE = int(2500)

---

In [3]:

engine = get_engine_from_env()


---

In [4]:


rebuild_result = rebuild_consumed_messages_to_observations(
    engine=engine,
    schema=SCHEMA,
    source_table=REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME,
    target_table=REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    rebuild_status=REBUILD_STATUS,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    mark_source_rebuilt=MARK_SOURCE_REBUILT_FLAG,
    
    observation_window_size=OBSERVATION_WINDOW_SIZE,
)

display(rebuild_result)

[obs-window] 1 | observation_index 1 to 2,500
[rebuild] Added 52 new columns to capstone.synthetic_sensor_observations_rebuilt_stage
[obs-window] 2 | observation_index 2,501 to 5,000
[obs-window] 3 | observation_index 5,001 to 7,500
[obs-window] 4 | observation_index 7,501 to 10,000
[obs-window] 5 | observation_index 10,001 to 12,500
[obs-window] 6 | observation_index 12,501 to 15,000
[obs-window] 7 | observation_index 15,001 to 17,500
[obs-window] 8 | observation_index 17,501 to 20,000
[obs-window] 9 | observation_index 20,001 to 22,500
[obs-window] 10 | observation_index 22,501 to 25,000
[obs-window] 11 | observation_index 25,001 to 27,500
[obs-window] 12 | observation_index 27,501 to 30,000
[obs-window] 13 | observation_index 30,001 to 32,500
[obs-window] 14 | observation_index 32,501 to 35,000
[obs-window] 15 | observation_index 35,001 to 37,500
[obs-window] 16 | observation_index 37,501 to 40,000
[obs-window] 17 | observation_index 40,001 to 42,500
[obs-window] 18 | observation_in

{'status': 'rebuilt',
 'consumed_rows': 3744000,
 'deduped_rows': 3744000,
 'rebuilt_rows': 72000,
 'rebuilt_observations': 72000,
 'updated_source_observation_groups': 72000,
 'target_table': 'synthetic_sensor_observations_rebuilt_stage'}

----

In [ ]:
validation_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        COUNT(*) AS rebuilt_row_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS complete_row_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        MIN(observation_timestamp) AS min_observation_timestamp,
        MAX(observation_timestamp) AS max_observation_timestamp
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    """
)

display(validation_dataframe)

---

In [ ]:
sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        observation_timestamp,
        stream_state,
        phase,
        meta_episode_id,
        meta_primary_fault_type,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        rebuild_sensor_count,
        rebuild_is_complete
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    ORDER BY observation_index
    LIMIT 10
    """
)
display(sample_dataframe)

----

In [12]:
incomplete_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        rebuild_sensor_count,
        rebuild_is_complete,
        rebuild_notes
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    WHERE rebuild_is_complete = FALSE
    ORDER BY observation_index
    LIMIT 25
    """
)

display(incomplete_dataframe)

,dataset_id,run_id,asset_id,observation_index,rebuild_sensor_count,rebuild_is_complete,rebuild_notes


----

In [ ]:
# Dispose Engine
engine.dispose()